# 06b · European split

The `06_Split` loop for the European form: portfolio search, capture into the **European** ledger (`Outputs/Euro/Analysis`, run codes `E001`, `E002`, ...) and the Edge Book. The league ledger is never written to.

In [ ]:
import fpp
import pandas as pd
from fpp import portfolio as pf
from fpp.europe import run as er

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

In [ ]:
ODDS_FILE = er.latest_euro_form()     # newest Outputs/Euro/odds_input_*.xlsx
STAKE     = 330.0
# Half the priced top-5 v top-5 fixtures, rounded up: how many a portfolio may leave out.
N_EVENTS  = fpp.report.read_filled(ODDS_FILE).pipe(lambda f: f[f[list(fpp.staking.BOOK_COLUMNS)].gt(0).any(axis=1)]['sheet_code'].nunique())
LEG_VAR   = -(-N_EVENTS // 2)
print(f'{N_EVENTS} priced fixtures -> LEG_VAR = {LEG_VAR}')

In [ ]:
meta = fpp.report.check_form_is_current(ODDS_FILE)
print(f"form      : {meta['path'].name}")
print(f"generated : {meta['generated']}  from {meta['source']}")
filled = fpp.report.read_filled(ODDS_FILE)

In [ ]:
res = pf.search(filled, leg_var=LEG_VAR)
info, scored = res["info"], res["scored"]
print(f"mode        : {info['mode']}  ({info['found']:,} reached -> pool {info['pool']:,})")
print(f"undominated : {int(scored['undominated'].sum()):,}")

In [ ]:
picked = pf.filter_portfolios(scored, min_pct_expected_return=1.05, min_p_over_100=0.70,
                              max_pct_sd=0.25, sort_by="p_over_100")
VIEW = ["id", "split", "legs", "median_leg_p", "pct_expected_return", "pct_sd", "p_over_100"]
display(picked[VIEW].head(15))

In [ ]:
slate = er.capture(res, filled, source=ODDS_FILE.name)
print("->", slate)
book = er.write_book(res, filled, source=ODDS_FILE.name, run_code=slate.run_code)
print("->", book)

## Settle

Once the matches are played: absorb pending European slates, refetch the European scoreboards for the dates still waiting, and settle.

In [ ]:
er.settle()
fpp.ledger.read_table("propositions", fpp.europe.config.EURO_ANALYSIS).groupby("league_key")["won"].agg(["count", "mean"])